In [ ]:
from typing import NamedTuple, Self
from jaxtyping import Array, Float, Scalar
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
import jax.scipy as jsp
import jax.random as jr
import equinox as eqx

import numpy as np
import scipy as sp

jax.config.update("jax_enable_x64", True)
EPS = float(jnp.sqrt(jnp.finfo(float).eps))

# RKHS Functions

In [ ]:
def kernel(x1: Float[Array, "n d"], x2: Float[Array, "m d"]) -> Float[Array, "n m"]:
    """Squared exponential kernel"""
    rho = 1 / (4 * jnp.pi) ** 2  # lengthscale of the kernel

    def d2(a: Float[Array, "d"], b: Float[Array, "d"]) -> Scalar:
        return jnp.sum(jnp.square(a - b) / rho)

    d2 = jax.vmap(jax.vmap(d2, in_axes=(None, 0)), in_axes=(0, None))
    k = jnp.exp(-0.5 * d2(x1, x2))
    return k

In [ ]:
class RKHSFunction(NamedTuple):
    a: Float[Array, "k"]  # coefficients
    x: Float[Array, "k d"]  # basis points

    @jax.jit
    def __call__(self, t: Float[Array, "n d"]) -> Float[Array, "n"]:
        Ktx = kernel(t, self.x)
        return Ktx @ self.a

    @classmethod
    def sample(cls, n: int, k: int, d: int) -> list[Self]:
        lhs_sampler = sp.stats.qmc.LatinHypercube(
            d=k * (d + 1), seed=np.random.mtrand._rand
        )
        ps = lhs_sampler.random(n=n).reshape(n, k, (d + 1))
        return [cls.from_array(pi) for pi in ps]

    @classmethod
    def from_array(
        cls,
        p: Float[Array, "k (d+1)"],
        x_range: tuple[float, float] = (0.0, 1.0),
        a_range: tuple[float, float] = (-1.0, 1.0),
        eps: float = 0.001,
    ) -> Self:
        x, a = p[:, :-1], p[:, -1]

        # ensure x is unique
        perm = jnp.argsort(x.flatten(), axis=0)
        x, a = x[perm], a[perm]
        x = x + jnp.arange(len(x))[:, None] * eps
        x = x / (1.0 + len(x) * eps)  # [0,1+n*eps] -> [0,1]
        # x = x * (1 - 2 * eps) + eps  # [0,1] -> [eps,1-eps]

        # scale into correct range
        x = x * (x_range[1] - x_range[0]) + x_range[0]  # [0,1]->x_range
        a = a * (a_range[1] - a_range[0]) + a_range[0]  # [0,1]->a_range
        return cls(a=a, x=x)


def linear_combination(basis_fs: list[RKHSFunction], coefficients: Float[Array, "n"]):
    x = [fi.x for fi in basis_fs]
    a = [fi.a * ci for fi, ci in zip(basis_fs, coefficients)]
    return RKHSFunction(x=jnp.concat(x), a=jnp.concat(a))

In [ ]:
def test_fn(x: Float[Array, "m 1"]) -> Float[Array, "m"]:
    return jnp.sinc(x * 2 * jnp.pi).squeeze()


def target_fn(f: RKHSFunction, n: int = 100000) -> Scalar:
    x = jnp.linspace(0, 1, n).reshape(-1, 1)
    return jnp.mean(jnp.square(f(x) - test_fn(x)))


def plot_fns(
    fs: list[RKHSFunction] = [],
    background_fs: list[RKHSFunction] = [],
) -> None:
    x = jnp.linspace(0, 1, 1000).reshape(-1, 1)
    plt.plot(x, test_fn(x), label="target", color="black", linestyle="dashed")
    for i, f in enumerate(background_fs):
        plt.plot(x, f(x), color="gray", alpha=0.3)
        plt.plot(f.x, f(f.x), "x", color="gray", alpha=0.1)
    for i, f in enumerate(fs):
        plt.plot(x, f(x))
        plt.plot(f.x, f.a, "x", color=plt.gca().lines[-1].get_color())
    plt.legend()
    plt.grid()
    plt.show()

# Fit GP

In [ ]:
class Gaussian(NamedTuple):
    mean: Float[Array, "n"]
    cov: Float[Array, "n n"]


class GaussianProcess(NamedTuple):
    # model parameters
    rho: Scalar  # lengthscale
    g: Scalar  # nugget
    nu: Scalar  # covariance scale
    b: Scalar  # trend

    # observed data
    observed_fs: list[RKHSFunction]
    observed_ys: Float[Array, "n"]

    # cached covariance matrix of the observed data
    Koo: Float[Array, "n n"]

    @classmethod
    def fit(
        cls,
        fs: list[RKHSFunction],
        ys: Float[Array, "n"],
        *,
        lengthscale_range: tuple[float, float] = (EPS, 100.0),
        nugget_range: tuple[float, float] = (EPS, 1e-3),
        max_iterations: int = 100,
    ):
        # define the loss function for optimization
        def loglikelihood(
            rho: Scalar, g: Scalar, fs: list[RKHSFunction], ys: Float[Array, "n"]
        ) -> tuple[Scalar, Scalar, Scalar]:
            # foward of kernel
            K = cls.kernel(rho, fs, fs)
            K = K + jnp.eye(len(ys)) * g

            # cholesky of K and compute logdet
            K_sqrt, is_lower = jsp.linalg.cho_factor(K)
            logdetK = 2.0 * jnp.sum(jnp.log(jnp.diag(K_sqrt)))

            # compute Ki_1=(K^-1 @ 1) and Ki_y=(K^-1 @ y)
            Ki_1, Ki_y = jsp.linalg.cho_solve(
                c_and_lower=(K_sqrt, is_lower),
                b=jnp.stack([jnp.ones_like(ys), ys], 1),
            ).T

            # compute optimal trend b and scale nu
            b = (Ki_1 * ys).sum() / Ki_1.sum()
            nu = jnp.dot((ys - b) / len(ys), (Ki_y - Ki_1 * b))

            # likelihood when marginalizing over trend and variance
            loglik = -0.5 * (len(ys) * jnp.log(nu) + logdetK)
            return loglik, b, nu

        @jax.jit
        @jax.value_and_grad
        def mle_loss(params: Float[Array, "2"]):
            rho, g = params[0], params[-1]
            loglik, b, nu = loglikelihood(rho, g, fs, ys)
            return -loglik

        # initialization
        nugget = min(0.1, nugget_range[1])
        lengthscale = 0.9 * lengthscale_range[1] + 0.1 * lengthscale_range[0]
        init_params = jnp.array([jnp.log(lengthscale), jnp.log(nugget)])

        # run optimization
        result = sp.optimize.minimize(
            fun=mle_loss,
            x0=init_params,
            jac=True,
            method="L-BFGS-B",
            bounds=[lengthscale_range, nugget_range],
            options=dict(maxiter=max_iterations, ftol=EPS, gtol=0),
        )

        # extract the optimal parameters and infer the rest
        rho = jnp.array(result.x[0])
        g = jnp.array(result.x[1])
        llk, b, nu = loglikelihood(rho, g, fs, ys)

        # precompute the covariance matrix of the observed data for faster predictions
        Koo = nu * (cls.kernel(rho, fs, fs) + g * jnp.eye(len(ys)))
        return cls(
            rho=rho, g=g, nu=nu, b=b, observed_fs=fs, observed_ys=ys, Koo=Koo
        )

    @staticmethod
    def kernel(
        rho: Scalar, fs_1: list[RKHSFunction], fs_2: list[RKHSFunction]
    ) -> Float[Array, "n m"]:
        def rkhs_dist_squared(f1: RKHSFunction, f2: RKHSFunction) -> Scalar:
            K11 = kernel(f1.x, f1.x)
            K12 = kernel(f1.x, f2.x)
            K22 = kernel(f2.x, f2.x)
            d2 = +f1.a @ K11 @ f1.a + f2.a @ K22 @ f2.a - 2 * f1.a @ K12 @ f2.a
            return jnp.maximum(d2, 0.0)  # ensure non-negativity

        # stack list of pytrees into a single pytree of arrays
        d2 = jnp.array([[rkhs_dist_squared(f1, f2) for f2 in fs_2] for f1 in fs_1])
        k = jnp.exp(-0.5 * d2 / rho)
        return k

    @jax.jit
    def predict(self, fs: list[RKHSFunction]) -> Gaussian:
        n = len(self.observed_ys)

        # compute covariance matrices
        Kxx = self.nu * self.kernel(self.rho, fs, fs)
        Kxo = self.nu * self.kernel(self.rho, fs, self.observed_fs)
        Koo = self.Koo

        # posterior mean and covariance
        mean = self.b + Kxo @ jnp.linalg.solve(Koo, self.observed_ys - self.b)
        cov = Kxx - Kxo @ jnp.linalg.solve(Koo, Kxo.T)

        # Add correction based on the trend estimation correlation
        Kbx = jnp.ones((1, n)) @ jnp.linalg.solve(Koo, Kxo.T)
        cov = cov + (1 - Kbx).T @ (1 - Kbx) / jnp.linalg.inv(Koo).sum()
        return Gaussian(mean=mean, cov=cov)

    @jax.jit
    def log_expected_improvement(self, f: RKHSFunction) -> Scalar:
        # numerically stable version following https://arxiv.org/pdf/2310.20708:
        mu, cov = self.predict([f])
        mu = mu.squeeze()
        sigma = cov.squeeze() ** 0.5
        z = (self.observed_ys.min() - mu) / sigma

        # use lax.cond to avoid propagating NaNs in the gradients
        branch1 = lambda: jnp.log(z * jsp.stats.norm.cdf(z) + jsp.stats.norm.pdf(z))
        branch2a = lambda: -2 * jnp.log(-z)
        branch2b = lambda: jax.nn.log1mexp(
            -jnp.log(-z)
            - jsp.stats.norm.logsf(-z)
            - z**2 / 2
            - jnp.log(2 * jnp.pi) / 2.0
        )
        branch2 = lambda: (
            -(z**2) / 2
            - jnp.log(2 * jnp.pi) / 2
            + jax.lax.cond(z < -1 / jnp.sqrt(EPS), branch2a, branch2b)
        )
        ei = jnp.log(sigma) + jax.lax.cond(z > -1, branch1, branch2)
        return ei

# Expected Improvement

In [ ]:
class BFGS(NamedTuple):
    multi_starts: int
    raw_samples: int = 1024
    max_iterations: int = 100
    ftol: float = EPS
    gtol: float = 0.0

    def __call__(
        self,
        model: GaussianProcess,
        basis_fs: list[RKHSFunction],
    ) -> RKHSFunction:

        # define the acquisition function and its gradient as a vect
        @jax.jit
        @jax.value_and_grad
        def loss(c: Float[Array, "n"]):
            f = linear_combination(basis_fs, c)
            return -model.log_expected_improvement(f)

        # sample initial guess for candidate points
        lhs_sampler = sp.stats.qmc.LatinHypercube(
            d=len(basis_fs), seed=np.random.mtrand._rand
        )
        xs = lhs_sampler.random(n=self.raw_samples)

        # keep the most promising initial guesses
        losses = np.array([loss(xi)[0] for xi in xs])
        xs = xs[np.argsort(losses)[: self.multi_starts]]

        # optimize each initial guess with L-BFGS
        results = [
            sp.optimize.minimize(
                fun=loss,
                x0=xi,
                jac=True,
                method="L-BFGS-B",
                bounds=[(-1.0, 1.0)] * len(basis_fs),
                options=dict(
                    maxiter=self.max_iterations,
                    ftol=self.ftol,
                    gtol=self.gtol,
                ),
            )
            for xi in xs
        ]

        # sort results and return best q
        losses = np.array([result.fun for result in results])
        xs = np.array([result.x for result in results])
        best_x = xs[np.argmin(losses)]
        return linear_combination(basis_fs, best_x)

In [ ]:
SUBSPACE_INITIAL_DIM = 5
SUBSPACE_FINAL_DIM = 10
BASIS_K = 3
ACQUISITION_EACH_DIM = 10

fs = RKHSFunction.sample(n=SUBSPACE_INITIAL_DIM, k=3, d=1)
ys = jnp.array([target_fn(fi) for fi in fs])
models = [GaussianProcess.fit(fs, ys)]

for dim in range(SUBSPACE_INITIAL_DIM, SUBSPACE_FINAL_DIM + 1):
    print(f"Acquisition with subspace dim = {dim}")
    acquisition = BFGS(multi_starts=16 * dim)

    for i in range(ACQUISITION_EACH_DIM):
        basis_fs = RKHSFunction.sample(n=dim, k=BASIS_K, d=1)
        f = acquisition(models[-1], basis_fs=basis_fs)
        y = target_fn(f)

        fs = fs + [f]
        ys = jnp.concatenate([ys, y[None]])
        models.append(GaussianProcess.fit(fs, ys))

        best_idx = jnp.argmin(ys)
        print(f"Iteration {i+1}: current= {y:.8f}, best = {ys[best_idx]:.8f}")

    best_idx = jnp.argmin(ys)
    plot_fns(
        fs=[fs[best_idx]],
        background_fs=fs[-ACQUISITION_EACH_DIM:],
    )

In [ ]:
plt.figure(figsize=(5, 3))
n0, dn = SUBSPACE_FINAL_DIM, ACQUISITION_EACH_DIM 
plt.plot(range(0, n0), ys[:n0], "o", label="basis fn")
for i, degree in enumerate(range(SUBSPACE_INITIAL_DIM, SUBSPACE_FINAL_DIM + 1)):
    y_deg = ys[n0 + i * dn : n0 + (i + 1) * dn]
    plt.plot(range(n0 + i * dn, n0 + (i + 1) * dn), y_deg, "o", label=f"acquired samples (dim={degree})")
plt.yscale("log")
plt.xlabel("Total Evaluations")
plt.ylabel("Target fn")
plt.title("Convergence of Bayesian Optimization")
plt.grid()
plt.legend(loc="center left", bbox_to_anchor=(1, 0.5))
plt.show()